# Phase 2 — YoY-log demand regression

Notebook 03 ran OLS on first-differenced monthly data and produced an R² of 0.09 with no individually-significant regressors — the symptom of shared annual seasonality surviving $\Delta_1$ differencing. Here we rebuild on **year-over-year log change**, $\Delta_{12} \log x = \log x_t - \log x_{t-12}$, which removes the calendar cycle and keeps the slow macro signal intact.

Sequence:

1. Build YoY-log series for arrivals and the three real-FX rates.
2. ADF on the YoY series to confirm stationarity.
3. OLS with Newey–West HAC (4 lags), Trends entering at lag 1.
4. Re-fit with month-of-year fixed effects; compare $R^2$.
5. Economic reading of the real-EUR/TRY elasticity.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_monthly.csv',
                 index_col=0, parse_dates=True)
df.index.name = 'date'
print(df.shape)

(190, 25)


## 1. YoY log-change transform

$yoy\_x_t = \log x_t - \log x_{t-12}$, expressed as a decimal (0.10 ≈ +10% YoY). Applied to arrivals and to the three real-FX series.

In [2]:
def yoy_log(s: pd.Series) -> pd.Series:
    return np.log(s) - np.log(s.shift(12))

df['yoy_arrivals']     = yoy_log(df['arrivals_total'])
df['yoy_real_EUR_TRY'] = yoy_log(df['real_EUR_TRY'])
df['yoy_real_GBP_TRY'] = yoy_log(df['real_GBP_TRY'])
df['yoy_real_USD_TRY'] = yoy_log(df['real_USD_TRY'])

yoy_cols = ['yoy_arrivals', 'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_USD_TRY']
df[yoy_cols].describe().round(3)

,yoy_arrivals,yoy_real_EUR_TRY,yoy_real_GBP_TRY,yoy_real_USD_TRY
count,106.000,171.000,171.000,177.000
mean,0.079,0.039,0.047,0.054
std,1.063,0.118,0.118,0.114
min,-5.057,-0.261,-0.237,-0.235
25%,0.048,-0.027,-0.021,-0.023
50%,0.127,0.042,0.052,0.045
75%,0.276,0.119,0.124,0.151
max,3.790,0.385,0.392,0.412


## 2. ADF on the YoY series

Same ADF (constant, AIC-selected lag, 5% threshold) on `yoy_arrivals` and `yoy_real_EUR_TRY`.

In [3]:
def adf_row(name, s):
    s = s.dropna()
    stat, pval, used_lag, nobs, crit, _ = adfuller(s, regression='c', autolag='AIC')
    return {
        'series':     name,
        'n':          int(nobs),
        'used_lag':   int(used_lag),
        'adf_stat':   round(float(stat), 3),
        'p_value':    round(float(pval), 4),
        'crit_5pct':  round(float(crit['5%']), 3),
        'conclusion': 'stationary' if pval < 0.05 else 'non-stationary',
    }

pd.DataFrame([
    adf_row('yoy_arrivals',     df['yoy_arrivals']),
    adf_row('yoy_real_EUR_TRY', df['yoy_real_EUR_TRY']),
])

,series,n,used_lag,adf_stat,p_value,crit_5pct,conclusion
0,yoy_arrivals,103,2,-3.622,0.0054,-2.89,stationary
1,yoy_real_EUR_TRY,156,14,-2.084,0.2511,-2.88,non-stationary


## 3. OLS — base specification

$$ yoy\_arrivals_t = \alpha
  + \beta_1\, yoy\_rEUR/TRY_t + \beta_2\, yoy\_rGBP/TRY_t
  + \beta_3\, Trends^{DE}_{t-1} + \beta_4\, Trends^{GB}_{t-1}
  + \gamma_1\,covid + \gamma_2\,war + \gamma_3\,mideast + \varepsilon_t $$

$\beta_1$ is an elasticity: a 1% real TRY depreciation against EUR is associated with $\beta_1\%$ change in YoY arrivals.

In [4]:
model_df = df.assign(
    trends_DE_lag1 = df['trends_DE_Türkei_Urlaub'].shift(1),
    trends_GB_lag1 = df['trends_GB_Turkey_holiday'].shift(1),
    month          = df.index.month,
)

regressors = [
    'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'covid', 'russia_war', 'mideast',
]
y_col = 'yoy_arrivals'

sample = model_df[[y_col, 'month'] + regressors].dropna()
print(f'Sample: {sample.index.min().date()} → {sample.index.max().date()}  (N={len(sample)})')

X_base = sm.add_constant(sample[regressors])
y_base = sample[y_col]
model_base = sm.OLS(y_base, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(model_base.summary())
print()
print(f'Durbin–Watson: {durbin_watson(model_base.resid):.3f}')

Sample: 2017-01-01 → 2025-03-01  (N=99)
                            OLS Regression Results                            
Dep. Variable:           yoy_arrivals   R-squared:                       0.218
Model:                            OLS   Adj. R-squared:                  0.157
Method:                 Least Squares   F-statistic:                     5.385
Date:                Thu, 21 May 2026   Prob (F-statistic):           3.46e-05
Time:                        15:43:00   Log-Likelihood:                -137.30
No. Observations:                  99   AIC:                             290.6
Df Residuals:                      91   BIC:                             311.4
Df Model:                           7                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------


## 4. Add month fixed effects, compare R²

Month-of-year dummies absorb any residual calendar effect not removed by the YoY transform (e.g., shifting Eid/Easter, school-break timing). The comparison cell prints $R^2$ and the real-EUR/TRY coefficient under each spec.

In [5]:
formula = (
    'yoy_arrivals ~ yoy_real_EUR_TRY + yoy_real_GBP_TRY'
    ' + trends_DE_lag1 + trends_GB_lag1'
    ' + covid + russia_war + mideast'
    ' + C(month)'
)
model_fe = smf.ols(formula, data=sample).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(model_fe.summary())
print()
print(f'Durbin–Watson: {durbin_watson(model_fe.resid):.3f}')

                            OLS Regression Results                            
Dep. Variable:           yoy_arrivals   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.054
Method:                 Least Squares   F-statistic:                     8.635
Date:                Thu, 21 May 2026   Prob (F-statistic):           2.68e-12
Time:                        15:43:00   Log-Likelihood:                -136.63
No. Observations:                  99   AIC:                             311.3
Df Residuals:                      80   BIC:                             360.6
Df Model:                          18                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.4779      0.413  

In [6]:
comp = pd.DataFrame({
    'no_FE': {
        'R2':         round(model_base.rsquared, 3),
        'Adj_R2':     round(model_base.rsquared_adj, 3),
        'beta_EUR':   round(model_base.params['yoy_real_EUR_TRY'], 3),
        'p_EUR':      round(model_base.pvalues['yoy_real_EUR_TRY'], 4),
        'DW':         round(durbin_watson(model_base.resid), 3),
    },
    'month_FE': {
        'R2':         round(model_fe.rsquared, 3),
        'Adj_R2':     round(model_fe.rsquared_adj, 3),
        'beta_EUR':   round(model_fe.params['yoy_real_EUR_TRY'], 3),
        'p_EUR':      round(model_fe.pvalues['yoy_real_EUR_TRY'], 4),
        'DW':         round(durbin_watson(model_fe.resid), 3),
    },
})
comp

,no_FE,month_FE
R2,0.2180,0.2280
Adj_R2,0.1570,0.0540
beta_EUR,-5.1560,-5.6580
p_EUR,0.1339,0.1398
DW,0.5290,0.5740


## 5. Economic interpretation — real-EUR/TRY elasticity

Cell below picks the **preferred spec** (higher adjusted-$R^2$) and translates its $\beta$ on `yoy_real_EUR_TRY` into the plain-English statement: *a 1% real TRY depreciation against EUR is associated with $\beta$% change in YoY arrivals*.

In [7]:
preferred_name, preferred = max(
    [('no-FE', model_base), ('month FE', model_fe)],
    key=lambda kv: kv[1].rsquared_adj,
)
b   = preferred.params['yoy_real_EUR_TRY']
p   = preferred.pvalues['yoy_real_EUR_TRY']
ci  = preferred.conf_int().loc['yoy_real_EUR_TRY']
alpha = 0.05

print(f'Preferred spec: {preferred_name}  (adj R² = {preferred.rsquared_adj:.3f})')
print(f'beta_EUR = {b:+.3f}   p = {p:.4f}   95% CI = [{ci[0]:+.3f}, {ci[1]:+.3f}]')
print()
verdict = 'significant at 5%' if p < alpha else 'NOT significant at 5%'
print(f'Result: real-EUR/TRY coefficient is {verdict}.')
print()
# In a log–log spec, beta is itself the elasticity:
# yoy_arrivals and yoy_real_EUR/TRY are both log-decimals, so a +1pp change
# in the regressor (i.e. Δ = 0.01) maps to a beta * 0.01 change in y, which
# reads as beta% in the dependent variable. So we print beta directly.
if p < alpha and b > 0:
    print('Economic reading:')
    print(f'  A 1% real TRY depreciation against EUR (yoy_real_EUR/TRY rises by 0.01)')
    print(f'  is associated with a {b:+.2f}% change in YoY arrivals.')
    print(f'  95% CI on that elasticity: [{ci[0]:+.2f}%, {ci[1]:+.2f}%].')
    print('  Sign matches the textbook substitution channel: cheaper TRY → more EU arrivals.')
elif p < alpha and b < 0:
    print(f'Sign is negative — a 1% real TRY depreciation is associated with {b:+.2f}% YoY arrivals.')
    print('  Counter to the textbook prior; likely picking up reverse-causal or omitted-variable dynamics.')
else:
    print(f'Point estimate: 1% real TRY depreciation ↔ {b:+.2f}% YoY arrivals.')
    print(f'  But the 95% CI [{ci[0]:+.2f}%, {ci[1]:+.2f}%] straddles zero — the data')
    print('  cannot rule out no FX effect at conventional levels. Interpret with caution.')

print()
print('Other regressors at 5%:')
sig = preferred.pvalues[preferred.pvalues < alpha].drop(['yoy_real_EUR_TRY'], errors='ignore')
if sig.empty:
    print('  (no other regressor clears p < 0.05)')
else:
    for name in sig.index:
        if name.startswith('C(month)'): continue
        print(f'  {name:25s}  beta={preferred.params[name]:+.3e}  p={sig[name]:.4f}')

Preferred spec: no-FE  (adj R² = 0.157)
beta_EUR = -5.156   p = 0.1339   95% CI = [-11.897, +1.586]

Result: real-EUR/TRY coefficient is NOT significant at 5%.

Point estimate: 1% real TRY depreciation ↔ -5.16% YoY arrivals.
  But the 95% CI [-11.90%, +1.59%] straddles zero — the data
  cannot rule out no FX effect at conventional levels. Interpret with caution.

Other regressors at 5%:
  yoy_real_GBP_TRY           beta=+6.952e+00  p=0.0405
  russia_war                 beta=+3.446e-01  p=0.0401
